Chama os notebooks de api para verificar se o cep solicitado já existe na base ou não

In [0]:
# CEP e numero informados pelo analista
dbutils.widgets.text("cep", "08717310", "CEP")
dbutils.widgets.text("numero", "123", "Numero")

cep = dbutils.widgets.get("cep")
numero = dbutils.widgets.get("numero")

# Executa os dois notebooks de extracao
dbutils.notebook.run("../src/01_api_viaCep", 120, {"cep": cep})
dbutils.notebook.run("../src/02_api_brasilapi", 120, {"cep": cep})

print("Consultas das APIs concluidas!")

Consulta as tabelas de cep após execução da api (função Fcn_Logradouro)

In [0]:
%sql
SELECT *
FROM geocep.default.Fcn_Logradouro(
    :cep,
    :numero
);

In [0]:
from pyspark.sql.functions import current_timestamp, year, month

# Aproveita o resultado da função SQL executada anteriormente
df = _sqldf

# Adiciona data do processamento e campos para particionamento
df = (
    df.withColumn("data_processamento", current_timestamp())
      .withColumn("ano", year(current_timestamp()))
      .withColumn("mes", month(current_timestamp()))
)

# Grava o histórico sem apagar os registros anteriores
CAMINHO = "/Volumes/geocep/default/geocep/output/historico"

(
    df.write
    .mode("append")
    .option("compression", "snappy")
    .partitionBy("ano", "mes")
    .parquet(CAMINHO)
)

print("Histórico Parquet atualizado!")
display(df)

In [0]:
%skip
# Lê apenas a pasta do histórico
CAMINHO = "/Volumes/geocep/default/geocep/output/historico"

df_historico = spark.read.parquet(CAMINHO)

display(
    df_historico.orderBy(
        "data_processamento",
        ascending=False
    )
)